In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

In [2]:
def ndcg_at_k(y_true, y_pred, k=22):
    """nDCG@Kを計算する関数"""
    def dcg(scores):
        return np.sum((2**scores - 1) / np.log2(np.arange(2, scores.size + 2)))

    k = min(k, len(y_true), len(y_pred))  # kをy_trueとy_predの長さ以内に制限

    if k == 0:
        return 0  # kが0の場合はnDCGを0として扱う
    
    # y_trueに基づいて上位k個のインデックスを取得
    order = np.argsort(y_pred)[::-1][:k]
    ideal_order = np.argsort(y_true)[::-1][:k]

    # 範囲外のインデックスを除外
    order = order[order < len(y_true)]
    ideal_order = ideal_order[ideal_order < len(y_true)]
    
    return dcg(y_true[order]) / (dcg(y_true[ideal_order]) + 1e-10)

In [3]:
train_df = pd.read_csv('../001.preprocess/train/train_A.csv')
test_df = pd.read_csv('../001.preprocess/test/test_A.csv')

In [4]:
# カテゴリ変数のエンコード
le_user = LabelEncoder()
le_product = LabelEncoder()
train_df["user_id"] = le_user.fit_transform(train_df["user_id"])
train_df["product_id"] = le_product.fit_transform(train_df["product_id"])

test_df.loc[:, "user_id"] = le_user.transform(test_df["user_id"])

In [5]:
#人気順の最大値を31にする
train_df["rank"] = train_df["rank"].clip(upper=31)

In [6]:
# 特徴量とターゲットの設定
X = train_df[["user_id", "product_id"]].values
y = train_df["rank"].values

In [7]:
# クロスバリデーションの設定（5-Fold）
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ndcg_scores = []
models = []

dparams = {
    "objective": "rank:pairwise",
    "eval_metric": "ndcg",
    "tree_method": "hist",
    "max_depth": 6,
    "learning_rate": 0.1,
    "lambda": 1.0,
    "ndcg_exp_gain": True,  # Gainを加味した評価
    "subsample": 0.8,  # データのサンプリング
    "colsample_bytree": 0.8,  # 特徴量のサンプリング
    "gamma": 0.1  # 木の分岐に対するペナルティ
}

In [8]:
# KFoldで学習・評価
for fold, (train_idx, valid_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold + 1}")
    
    X_train, X_valid = X[train_idx], X[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    bst = xgb.train(dparams, dtrain, num_boost_round=100, evals=[(dvalid, "valid")], early_stopping_rounds=10)

    models.append(bst)

    # 予測とnDCG計算
    y_valid_pred = bst.predict(dvalid)
    ndcg = ndcg_at_k(y_valid, y_valid_pred)
    ndcg_scores.append(ndcg)
    print(f"nDCG@22 for fold {fold + 1}: {ndcg:.4f}")

Fold 1
[0]	valid-ndcg:0.00138
[1]	valid-ndcg:0.24360
[2]	valid-ndcg:0.39032
[3]	valid-ndcg:0.64423
[4]	valid-ndcg:0.45117
[5]	valid-ndcg:0.45117
[6]	valid-ndcg:0.50065
[7]	valid-ndcg:0.50065
[8]	valid-ndcg:0.61037
[9]	valid-ndcg:0.61037
[10]	valid-ndcg:0.61037
[11]	valid-ndcg:0.61037
[12]	valid-ndcg:0.61037
[13]	valid-ndcg:0.54881
nDCG@22 for fold 1: 0.5947
Fold 2
[0]	valid-ndcg:0.12935
[1]	valid-ndcg:0.26300
[2]	valid-ndcg:0.26300
[3]	valid-ndcg:0.29125
[4]	valid-ndcg:0.29125
[5]	valid-ndcg:0.29125
[6]	valid-ndcg:0.33293
[7]	valid-ndcg:0.44991
[8]	valid-ndcg:0.49445
[9]	valid-ndcg:0.47746
[10]	valid-ndcg:0.47746
[11]	valid-ndcg:0.47746
[12]	valid-ndcg:0.49510
[13]	valid-ndcg:0.46363
[14]	valid-ndcg:0.53868
[15]	valid-ndcg:0.53868
[16]	valid-ndcg:0.62726
[17]	valid-ndcg:0.62726
[18]	valid-ndcg:0.57637
[19]	valid-ndcg:0.57637
[20]	valid-ndcg:0.57637
[21]	valid-ndcg:0.57637
[22]	valid-ndcg:0.47978
[23]	valid-ndcg:0.49929
[24]	valid-ndcg:0.49929
[25]	valid-ndcg:0.48862
[26]	valid-ndcg:0.4

In [9]:
# クロスバリデーションの平均nDCG
print(f"Mean nDCG@22 across folds: {np.mean(ndcg_scores):.4f}")

Mean nDCG@22 across folds: 0.5059


In [10]:
# 最良のモデルで予測を行う
best_model = models[np.argmax(ndcg_scores)]

In [11]:
# 予測の生成
all_product_ids = train_df["product_id"].unique()

In [12]:
def generate_predictions_for_user(user):
    user_data = pd.DataFrame({"user_id": [user] * len(all_product_ids), "product_id": all_product_ids})
    dtest = xgb.DMatrix(user_data)
    user_preds = best_model.predict(dtest)
    user_pred_df = pd.DataFrame({"user_id": user, "product_id": all_product_ids, "pred_score": user_preds})
    user_pred_df.sort_values("pred_score", ascending=False, inplace=True)
    user_pred_df = user_pred_df.head(22)
    user_pred_df["rank"] = range(len(user_pred_df))
    
    return user_pred_df

In [13]:
predictions = pd.concat([generate_predictions_for_user(user) for user in test_df["user_id"].unique()])

In [14]:
# エンコードしたカテゴリ変数の戻し
predictions["user_id"] = le_user.inverse_transform(predictions["user_id"])
predictions["product_id"] = le_product.inverse_transform(predictions["product_id"])
predictions_drop = predictions.drop('pred_score', axis=1)

In [15]:
# 結果をファイルに出力
predictions_drop.to_csv('./predict/predictions_A.tsv', sep='\t', index=False)